<hr style="border: 6px solid#003262;" />

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/pytorch_logo.png" align="center" width="20%" padding="10px"><br>
</div>

<br>

# AN INTRODUCTION TO PYTORCH

<br>

**About:** PyTorch is an open-source machine learning library built on dynamic computation graphs and a NumPy-familiar tensor API. This notebook introduces PyTorch from the ground up - covering tensors, automatic differentiation with `autograd`, and building trainable models with `nn.Module` - so that readers can move from raw tensor math to a working training loop.

**Learning Goals:**
- Create and manipulate PyTorch tensors across scalar, vector, matrix, and higher-rank shapes
- Use tensor operations (arithmetic, indexing, slicing, reshape) and understand how they differ from NumPy
- Enable gradient tracking with `requires_grad` and compute gradients using `.backward()`
- Build a neural network by subclassing `nn.Module` and implementing `forward`
- Write a complete training loop using a loss function and an optimizer
- Understand when to use `model.eval()` and `torch.no_grad()` during inference

**Keywords:** pytorch, pytorch-tutorial, deep-learning, autograd, neural-networks, nn-module

**Prerequisite Knowledge:** (1) Python, (2) NumPy, (3) Linear Algebra

**Target User:** Data scientists, applied machine learning engineers, and developers new to PyTorch

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 0: ABOUT AND MOTIVATION](#Part_0)
> #### [PART 1: TENSORS AND OPERATIONS](#Part_1)
> #### [PART 2: AUTOGRAD - AUTOMATIC DIFFERENTIATION](#Part_2)
> #### [PART 3: NEURAL NETWORKS WITH nn.Module](#Part_3)
> #### [PART 4: WRAP UP AND NEXT STEPS](#Part_4)

#### APPENDIX

> #### [PYTORCH INSTALLATION](#Appendix_1)
> #### [REFERENCES AND ADDITIONAL RESOURCES](#Appendix_2)

<br>

<a id='Part_0'></a>

<hr style="border: 2px solid#003262;" />

#### PART 0

## **ABOUT** AND **MOTIVATION**

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/pytorch_logo.png" align="center" width="30%" padding="10px"><br>
    <br>
</div>

[**PyTorch**](https://pytorch.org/) is a dynamic, define-by-run deep learning framework. Unlike older static-graph frameworks, PyTorch builds its computation graph on the fly as Python executes - which means you can use ordinary Python control flow (`if`, `for`, `while`) and inspect intermediate tensor values at any point. This makes debugging feel natural and is one of the main reasons PyTorch became the dominant framework for deep learning research.

PyTorch and TensorFlow 2.x have converged significantly - both support eager execution by default - but they differ in philosophy, API design, and community use patterns. The table below summarizes the most relevant comparisons for a learner coming from either framework:

| Feature | PyTorch | TensorFlow 2.x |
|---|---|---|
| Execution mode | Eager by default | Eager by default |
| Gradient computation | `loss.backward()` + `.grad` | `tf.GradientTape` |
| Model building | `nn.Module` subclass | `tf.keras.Model` or `nn.Sequential` |
| Graph compilation | `torch.compile()` (PyTorch 2.0+) | `@tf.function` |
| Dominant use case | Research, academic papers | Production, industry deployment |

<strong style="color:red">KEY CONSIDERATION:</strong> Some shell commands in the Setup section use bash syntax and require a Unix environment (Linux or macOS). Windows users should run these in WSL2 or adjust the commands to PowerShell syntax.

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **TENSORS** AND **OPERATIONS**

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/pytorch_logo.png" align="center" width="20%" padding="10px"><br>
    <br>
</div>

#### CONTENTS:

> [PART 1.1: PYTORCH SETUP](#Part_1_1)<br>
> [PART 1.2: TENSOR CREATION](#Part_1_2)<br>
> [PART 1.3: TENSOR OPERATIONS](#Part_1_3)<br>
> [PART 1.4: INDEXING, SLICING, AND RESHAPING](#Part_1_4)<br>

<a id='Part_1_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.1: PYTORCH SETUP

<br>

**Install PyTorch:** the recommended approach for most learners is pip inside a virtual environment. For GPU-specific wheels, use the [official install selector](https://pytorch.org/get-started/locally/) at pytorch.org.

In [ ]:
## Install PyTorch (CPU build - adjust for CUDA if you have a GPU) ##
# Verified against PyTorch 2.x install page, 2026-08-28 - re-check at https://pytorch.org/get-started/locally/
# ! pip install torch torchvision

In [ ]:
import torch
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import torch.nn as nn

%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

print('PyTorch version:', torch.__version__)

In [ ]:
# Check whether a CUDA GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

___

**Note:** All tensors and models in this notebook are created on CPU. Moving them to a GPU (if available) is covered in the optional section on device placement.

___

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_1_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.2: TENSOR CREATION

<br>

[**Tensors**](https://docs.pytorch.org/docs/2.13/tensors.html) are PyTorch's fundamental data structure - multi-dimensional arrays analogous to NumPy `ndarray`s, but with two key additions: they can live on a GPU, and they can participate in automatic differentiation. Every tensor has a `dtype`, a `shape`, and a `device`.

___

**Note:** `torch.tensor()` (lowercase `t`) is the general-purpose factory function that copies data into a new tensor. `torch.Tensor` (uppercase `T`) is the class itself. Prefer the factory function in practice.

___

<br>

**Scalar, vector, and matrix tensors**

In [ ]:
# Scalar (rank-0 tensor)
scalar = torch.tensor(4.0)
print('Scalar:', scalar)
print('  ndim:', scalar.ndim)
print('  shape:', scalar.shape)
print('  dtype:', scalar.dtype)

In [ ]:
# Vector (rank-1 tensor)
vector = torch.tensor([1.0, 2.0, 3.0])
print('Vector:', vector)
print('  shape:', vector.shape)

In [ ]:
# Matrix (rank-2 tensor) - explicit dtype
matrix = torch.tensor([[1, 2, 3],
                        [4, 5, 6]], dtype=torch.float32)
print('Matrix:\n', matrix)
print('  shape:', matrix.shape)

In [ ]:
# Rank-3 tensor
rank3 = torch.tensor([[[1, 2], [3, 4]],
                        [[5, 6], [7, 8]]], dtype=torch.float32)
print('Rank-3 shape:', rank3.shape)

<br>

**Factory functions for common patterns**

In [ ]:
# All zeros
zeros = torch.zeros(3, 4)
print('zeros(3,4):\n', zeros)

In [ ]:
# All ones
ones = torch.ones(2, 3)
print('ones(2,3):\n', ones)

In [ ]:
# Uniform random [0, 1)
rand_uniform = torch.rand(3, 3)
print('rand(3,3):\n', rand_uniform)

In [ ]:
# Standard normal
rand_normal = torch.randn(3, 3)
print('randn(3,3):\n', rand_normal)

In [ ]:
# Integer range - like Python range() but returns a tensor
arange = torch.arange(0, 10, step=2)
print('arange(0,10,2):', arange)

<br>

**Interoperability with NumPy**

In [ ]:
# NumPy array -> PyTorch tensor (shares memory on CPU)
arr = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(arr)
print('From NumPy:', t)

# Modifying arr also changes t - they share the same memory
arr[0] = 99.0
print('After modifying arr, t is:', t)

In [ ]:
# PyTorch tensor -> NumPy array
t2 = torch.tensor([4.0, 5.0, 6.0])
arr2 = t2.numpy()
print('As NumPy:', arr2, type(arr2))

___

**Note:** `torch.from_numpy()` creates a tensor that shares memory with the NumPy array - mutations to one propagate to the other. Use `.clone()` to break the shared memory link when you need an independent copy.

___

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_1_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.3: TENSOR OPERATIONS

<br>

#### **1.3.1 Arithmetic Operations**
___

In [ ]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.tensor([[10.0, 20.0], [30.0, 40.0]])

In [ ]:
# Element-wise addition
print(torch.add(a, b))
print(a + b)  # operator overloading

In [ ]:
# Element-wise multiplication
print(torch.mul(a, b))
print(a * b)

In [ ]:
# Matrix multiplication - two equivalent forms
print(torch.matmul(a, b))
print(a @ b)

___

**Note:** PyTorch distinguishes `torch.mul` (element-wise) from `torch.matmul` (matrix multiply). This distinction matters for gradients during backpropagation - always verify which one you need.

___

<br>

**Reduction operations**

In [ ]:
c = torch.tensor([[3.0, 7.0], [10.0, 2.0]])

print('Max:', torch.max(c))           # global max
print('Min:', torch.min(c))           # global min
print('Sum:', torch.sum(c))           # sum all elements
print('Mean:', torch.mean(c))         # mean
print('Argmax:', torch.argmax(c))     # flat index of max

In [ ]:
# Reduce along an axis
print('Max along rows (dim=0):', torch.max(c, dim=0).values)
print('Max along cols (dim=1):', torch.max(c, dim=1).values)

<br>

**Broadcasting**

In [ ]:
# PyTorch broadcasting follows NumPy rules exactly
x = torch.tensor([[1.0], [2.0], [3.0]])  # shape (3,1)
y = torch.tensor([10.0, 20.0, 30.0])     # shape (3,)

# y is broadcast to match x's column dimension
print('x shape:', x.shape)
print('y shape:', y.shape)
print('x + y:\n', x + y)

<br>

**In-place operations**

In [ ]:
# In-place operations are suffixed with underscore _ in PyTorch
d = torch.ones(2, 2)
d.add_(5)  # modifies d in place
print('After add_(5):', d)

___

**Note:** In-place operations (`add_`, `mul_`, etc.) modify the tensor without allocating a new one, which saves memory. However, they can interfere with gradient computation - avoid in-place modifications on tensors that have `requires_grad=True`.

___

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_1_4'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.4: INDEXING, SLICING, AND RESHAPING

<br>

PyTorch indexing follows standard Python/NumPy conventions. Integer indexing removes a dimension; slice indexing preserves it.

In [ ]:
t = torch.arange(12).reshape(3, 4).float()
print('t:\n', t)

In [ ]:
# Integer indexing (removes dimension)
print('Row 0:', t[0])          # shape (4,)
print('Element [1,2]:', t[1, 2])

In [ ]:
# Slice indexing (preserves dimension)
print('First two rows:\n', t[:2, :])    # shape (2, 4)
print('Last column:\n', t[:, -1:])      # shape (3, 1)

<br>

**Reshaping**

In [ ]:
# .view() - returns a tensor with a new shape sharing storage
# Requires contiguous memory layout
flat = t.view(12)
print('view(12):', flat.shape)

# .reshape() - works even if tensor is not contiguous (may copy)
reshaped = t.reshape(4, 3)
print('reshape(4,3):\n', reshaped)

In [ ]:
# -1 as a dimension lets PyTorch infer the size
auto_shaped = t.reshape(-1, 6)  # PyTorch infers first dim = 2
print('reshape(-1,6):', auto_shaped.shape)

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Create a rank-3 tensor of shape `(2, 3, 4)` filled with standard normal random values. Then: (a) retrieve the value at position `[1, 2, 3]` as a Python scalar, (b) extract the slice spanning all rows of the second matrix (`[1, :, :]`), and (c) reshape the entire tensor into shape `(6, 4)`. Store the results in `val`, `slice_t`, and `reshaped_t` respectively.**

<br>

```python
# starter code
import torch
t3 = torch.randn(2, 3, 4)

val       = ...
slice_t   = ...
reshaped_t = ...
```

<hr style="border: 2px solid#003262;" />

In [ ]:
# your code here


<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **AUTOGRAD** - AUTOMATIC **DIFFERENTIATION**

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/pytorch_logo.png" align="center" width="20%" padding="10px"><br>
    <br>
</div>

#### CONTENTS:

> [PART 2.1: requires_grad AND THE COMPUTATION GRAPH](#Part_2_1)<br>
> [PART 2.2: CALLING .backward()](#Part_2_2)<br>
> [PART 2.3: STOPPING GRADIENT TRACKING](#Part_2_3)<br>

<a id='Part_2_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.1: requires_grad AND THE COMPUTATION GRAPH

<br>

Every tensor operation in PyTorch is recorded in a dynamic computation graph (the `grad_fn` chain). When you flag a tensor with `requires_grad=True`, PyTorch tracks every operation that produces tensors downstream from it. Calling `.backward()` on a scalar output then walks this graph in reverse, accumulating the chain-rule product at each node into each leaf tensor's `.grad` attribute.

This mechanism - `autograd` - is what makes gradient descent automatic. You define the forward computation; PyTorch derives the gradients.

In [ ]:
# A leaf tensor with gradient tracking enabled
x = torch.tensor(3.0, requires_grad=True)
print('x:', x)
print('requires_grad:', x.requires_grad)
print('grad_fn:', x.grad_fn)  # None for leaf tensors

In [ ]:
# Any operation on x produces a non-leaf tensor that records its history
y = x ** 2
print('y = x^2:', y)
print('y.grad_fn:', y.grad_fn)  # PowBackward0 - records the power operation

In [ ]:
z = 2 * y + 5
print('z = 2y + 5:', z)
print('z.grad_fn:', z.grad_fn)  # AddBackward0

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.2: CALLING .backward()

<br>

`.backward()` computes $\partial z / \partial x$ for every leaf tensor in the graph. For $z = 2x^2 + 5$, the derivative is $dz/dx = 4x$. At $x = 3$, the expected gradient is $4 \cdot 3 = 12$.

In [ ]:
# Compute gradients by calling .backward() on the scalar output
z.backward()

# The gradient accumulates in x.grad
print('dz/dx at x=3:', x.grad)  # expected: 4*3 = 12

___

**Note:** Gradients accumulate in `.grad` across multiple `.backward()` calls - they are not reset automatically. In a training loop, always call `optimizer.zero_grad()` (or `tensor.grad.zero_()`) before computing new gradients, or gradients from previous steps will be added to the current ones.

___

In [ ]:
# Multi-variable gradient example
# f(u, v) = 3u^2 + uv + v^3  at u=2, v=1
# df/du = 6u + v = 13,  df/dv = u + 3v^2 = 5
u = torch.tensor(2.0, requires_grad=True)
v = torch.tensor(1.0, requires_grad=True)

f = 3 * u**2 + u * v + v**3
f.backward()

print('df/du:', u.grad)  # expected 13.0
print('df/dv:', v.grad)  # expected 5.0

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.3: STOPPING GRADIENT TRACKING

<br>

Gradient tracking has a memory cost - PyTorch stores intermediate activations in the computation graph so it can compute gradients during `.backward()`. During inference (prediction), you don't need gradients, so wrapping code in `torch.no_grad()` saves both memory and compute.

In [ ]:
# torch.no_grad() - the standard way to disable gradient tracking for a block
w = torch.tensor(5.0, requires_grad=True)

with torch.no_grad():
    result = w * 2
    print('requires_grad inside no_grad:', result.requires_grad)  # False

In [ ]:
# .detach() - creates a new tensor that shares storage but is removed from the graph
w2 = torch.tensor(5.0, requires_grad=True)
detached = w2.detach()
print('detached.requires_grad:', detached.requires_grad)  # False
print('w2.requires_grad still:', w2.requires_grad)        # True

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **For the function $h(x) = \sin(x^2) + \cos(x)$, compute $dh/dx$ at $x = \pi/4$ using PyTorch autograd. Store the gradient in `dh_dx`. Then verify your result by computing the analytical derivative $dh/dx = 2x\cos(x^2) - \sin(x)$ at the same point and storing it in `dh_dx_analytical`.**

<br>

```python
# starter code
import torch
import math

x = torch.tensor(math.pi / 4, requires_grad=True)

# compute h(x) and call backward
h = ...
...

dh_dx = x.grad
dh_dx_analytical = ...
```

<hr style="border: 2px solid#003262;" />

In [ ]:
# your code here


<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **NEURAL NETWORKS** WITH **nn.Module**

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/pytorch_logo.png" align="center" width="20%" padding="10px"><br>
    <br>
</div>

#### CONTENTS:

> [PART 3.1: BUILDING A MODEL WITH nn.Module](#Part_3_1)<br>
> [PART 3.2: THE TRAINING LOOP](#Part_3_2)<br>
> [PART 3.3: EVALUATING THE MODEL](#Part_3_3)<br>

<a id='Part_3_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.1: BUILDING A MODEL WITH nn.Module

<br>

[`nn.Module`](https://docs.pytorch.org/docs/main/notes/modules.html) is PyTorch's base class for all neural network models. Subclassing it gives you automatic parameter tracking (`.parameters()` returns all trainable tensors), device management (`.to(device)`), and save/load support. Every custom model follows the same two-method contract:

- `__init__`: define layers and any learnable components
- `forward`: describe how input flows through those layers

You never call `forward` directly - you call the model instance, and PyTorch calls `forward` for you while also running any registered hooks.

In [ ]:
class LinearRegressor(nn.Module):
    """Single linear layer: y = Wx + b"""

    def __init__(self, in_features, out_features):
        super().__init__()
        # nn.Linear holds W and b as learnable parameters
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.linear(x)


model = LinearRegressor(in_features=1, out_features=1)
print(model)

In [ ]:
# Inspect learnable parameters
for name, param in model.named_parameters():
    print(name, param.shape, param.requires_grad)

___

**Note:** All parameters registered through `nn.Linear`, `nn.Conv2d`, etc. have `requires_grad=True` by default. PyTorch's optimizer only updates tensors returned by `model.parameters()`, so parameters that are not registered as module attributes won't be trained.

___

<br>

**nn.Sequential - a shorthand for stacked layers**

In [ ]:
# For simple feedforward architectures, nn.Sequential is more concise
mlp = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)
print(mlp)

In [ ]:
# A forward pass - just call the model as a function
sample_input = torch.randn(5, 2)  # batch of 5, 2 features each
output = mlp(sample_input)
print('Input shape:', sample_input.shape)
print('Output shape:', output.shape)

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.2: THE TRAINING LOOP

<br>

Every PyTorch training loop follows the same four-step pattern per batch:

1. **Zero gradients** - `optimizer.zero_grad()` clears accumulated gradients from the previous step
2. **Forward pass** - call the model to get predictions
3. **Compute loss** - measure how far predictions are from targets
4. **Backward + update** - `loss.backward()` computes gradients, `optimizer.step()` applies them

The example below fits a simple linear model $\hat{y} = Wx + b$ to noisy data drawn from $y = 2x + 1$.

In [ ]:
# Generate noisy training data from y = 2x + 1
torch.manual_seed(42)
X_train = torch.rand(100, 1) * 4 - 2    # uniform in [-2, 2]
y_train = 2 * X_train + 1 + 0.3 * torch.randn(100, 1)

plt.scatter(X_train.numpy(), y_train.numpy(), alpha=0.5, label='data')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Training data: y = 2x + 1 + noise')
plt.legend()
plt.show()

In [ ]:
# Model, loss, optimizer
# Verified against PyTorch 2.x docs, 2026-08-28
reg_model = LinearRegressor(in_features=1, out_features=1)
criterion  = nn.MSELoss()
optimizer  = torch.optim.SGD(reg_model.parameters(), lr=0.05)

In [ ]:
# Training loop
epochs = 200
losses = []

for epoch in range(epochs):
    # 1. Zero gradients from last step
    optimizer.zero_grad()

    # 2. Forward pass
    predictions = reg_model(X_train)

    # 3. Compute loss
    loss = criterion(predictions, y_train)

    # 4. Backprop + update
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

    if epoch % 40 == 0:
        print(f'Epoch {epoch:>3d} | Loss: {loss.item():.4f}')

In [ ]:
# Plot loss curve
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training loss')
plt.show()

In [ ]:
# Learned parameters - should be close to W=2, b=1
W_learned = reg_model.linear.weight.item()
b_learned = reg_model.linear.bias.item()
print(f'Learned W: {W_learned:.4f}  (true: 2.0)')
print(f'Learned b: {b_learned:.4f}  (true: 1.0)')

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.3: EVALUATING THE MODEL

<br>

During evaluation you want two things:
- **No gradient computation** - saves memory, speeds up inference
- **Layers in eval mode** - disables training-specific behavior like Dropout and BatchNorm's running statistics

Use `model.eval()` + `torch.no_grad()` together. They serve different purposes: `eval()` flips behavior flags in layers; `no_grad()` stops PyTorch from building the computation graph.

In [ ]:
# Test data
X_test = torch.linspace(-2, 2, 50).unsqueeze(1)
y_test = 2 * X_test + 1

# Evaluation
reg_model.eval()
with torch.no_grad():
    y_pred = reg_model(X_test)
    test_loss = criterion(y_pred, y_test)

print(f'Test MSE: {test_loss.item():.4f}')

plt.scatter(X_train.numpy(), y_train.numpy(), alpha=0.3, label='train data')
plt.plot(X_test.numpy(), y_pred.numpy(), 'r-', linewidth=2, label='model')
plt.plot(X_test.numpy(), y_test.numpy(), 'g--', linewidth=1, label='true')
plt.legend()
plt.title('Linear regressor fit')
plt.show()

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Build a two-layer MLP classifier named `TwoLayerMLP` using `nn.Module`. It should accept 4 input features, pass them through a hidden layer of 8 units with ReLU activation, and produce 3 output logits. Instantiate it, confirm it has the right number of parameters, and run one forward pass on a batch of 10 random samples.**

<br>

```python
# starter code
import torch
import torch.nn as nn

class TwoLayerMLP(nn.Module):
    def __init__(self, in_features, hidden, out_features):
        super().__init__()
        # define layers here
        ...

    def forward(self, x):
        # define forward pass here
        ...

model_check = TwoLayerMLP(in_features=4, hidden=8, out_features=3)
sample = torch.randn(10, 4)
output = ...
#print('Output shape:', output.shape)  # expected: torch.Size([10, 3])
```

<hr style="border: 2px solid#003262;" />

In [ ]:
# your code here


<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **WRAP-UP** AND **NEXT STEPS**

<br>

You have completed the introduction to PyTorch. You now have a foundation in tensors, automatic differentiation, and building trainable models with `nn.Module`. To continue:

> [PyTorch Tutorials](https://docs.pytorch.org/tutorials/) - Beginner-to-advanced hands-on guides from the official team.<br>
> [torch.nn Documentation](https://docs.pytorch.org/docs/stable/nn.html) - Full list of layers, loss functions, and utilities.<br>
> [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/) - A higher-level framework that structures the training loop, removes boilerplate, and adds logging and hardware-agnostic training.<br>
> [torch.compile](https://docs.pytorch.org/docs/stable/generated/torch.compile.html) - PyTorch 2.0+'s graph compiler; analogous to TF's `@tf.function`.

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Appendix_1'></a>

<hr style="border: 6px solid#003262;" />

#### APPENDIX I

## **PYTORCH** INSTALLATION

<br>

#### **Appendix I.1 Create a Virtual Environment**
___

```bash
# Create and activate
python3 -m venv venv
source venv/bin/activate      # Linux / macOS
# venv\Scripts\activate       # Windows (PowerShell)
```

#### **Appendix I.2 Install PyTorch**
___

Always generate your install command from the [official selector](https://pytorch.org/get-started/locally/) at pytorch.org - the right wheel depends on your OS, CUDA version (if any), and Python version.

```bash
# CPU-only (Linux / macOS)
# Verified against pytorch.org install selector, 2026-08-28 - confirm at https://pytorch.org/get-started/locally/
pip install torch torchvision
```

```bash
# Verify install
python -c "import torch; print(torch.__version__); print(torch.cuda.is_available())"
```

<a id='Appendix_2'></a>

<hr style="border: 2px solid#003262;" />

#### APPENDIX II

## **REFERENCES** AND ADDITIONAL **RESOURCES**

<br>

> [PyTorch Official Tutorials](https://docs.pytorch.org/tutorials/)<br>
> [PyTorch Documentation - torch.Tensor](https://docs.pytorch.org/docs/stable/tensors.html)<br>
> [PyTorch Documentation - torch.autograd](https://docs.pytorch.org/docs/stable/autograd.html)<br>
> [PyTorch Documentation - torch.nn](https://docs.pytorch.org/docs/stable/nn.html)<br>
> [Deep Learning with PyTorch by Eli Stevens, Luca Antiga, and Thomas Viehmann](https://www.manning.com/books/deep-learning-with-pytorch)<br>
> [Programming PyTorch for Deep Learning by Ian Pointer](https://www.oreilly.com/library/view/programming-pytorch-for/9781492045342/)

<br>

*Sources Consulted: PyTorch 2.13 official documentation at docs.pytorch.org (retrieved 2026-08-28). API details verified against the stable release.*

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<hr style="border: 6px solid#003262;" />